# CFD-Drag-2D: CNN training extras (M10)

Trains the M5 CNN with mirror augmentation (`--flip`), Re-bin loss weights (`--re-weight-bins 8`)
and both, for seeds 0-2, and evaluates each run (metrics include Re bands and mirror consistency).
Output: `/kaggle/working/m10_cnn/<variant>_seed<k>/` with `model.pt`, `metrics.json`, `predictions.csv`.

In [ ]:
REPO = "https://github.com/PradeepkrishnaGM/CFD-Drag-2D.git"
COMMIT = "main"   # branch or commit SHA
EPOCHS = 150
SEEDS = [0, 1, 2]
VARIANTS = {"flip": "--flip", "reweight": "--re-weight-bins 8", "flip_reweight": "--flip --re-weight-bins 8"}
RUN = "/kaggle/working/m10_cnn"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/repo && git clone -q $REPO /tmp/repo && git -C /tmp/repo checkout -q $COMMIT && git -C /tmp/repo log --oneline -1

In [ ]:
%cd /tmp/repo
!pip install -q -e . --no-deps && pip install -q pytest
import glob, os, torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
DATA = os.path.dirname(sorted(glob.glob("/kaggle/input/**/cfd_drag_2d/data.csv", recursive=True) or glob.glob("/kaggle/input/**/data.csv", recursive=True))[0])
print("data:", DATA)
!python -m pytest -q tests/test_surrogate.py tests/test_m10_training.py -k "not mirrored" 2>&1 | tail -3

In [ ]:
for name, flags in VARIANTS.items():
    for seed in SEEDS:
        out = f"{RUN}/{name}_seed{seed}"
        !python scripts/train.py --data $DATA --out $out --epochs $EPOCHS --device cuda --seed $seed $flags
        !python scripts/evaluate.py --data $DATA --run $out | tail -6
        !git rev-parse HEAD > $out/commit.txt